# 04 输出格式对泄露的影响

## 实验概述

**目的：** 探究不同输出格式如何影响泄露的表现形式和可检测性。输出格式决定了模型"表达"记忆的方式——结构化格式（5-bin）迫使模型量化不确定性，而自由文本格式可能暴露更多质性泄露信号。

**四种输出格式：**
| 格式 | 输出要求 | PC 计算方式 | IDS 信号 |
|------|---------|------------|---------|
| binary | 涨/跌二分类 | I[ŷ_orig == ŷ_cf] | 无（二值无分布） |
| scalar | 浮点情绪分 | sign 一致性 | 低（一维信号） |
| 5-bin | 5 桶概率分布 | argmax 一致性 | 高（KL 散度捕捉分布形状） |
| free-text | 纯文本分析 | LLM-as-judge 提取方向 | 无 + 可检测质性泄露 |

**关键指标：** PC / CI / IDS / L（定义同 notebook 02）

**预期：** 5-bin 格式应提供最丰富的 IDS 信号，binary 格式因随机一致率高而 PC 灵敏度最低。

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.masking import extract_json_robust
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, load_counterfactual_variants
from src.experiment import run_scoring_batch
from src.masking import apply_masking
from src.prompts import scoring_prompt, timeliness_judge_prompt
from src.metrics import prediction_consistency, confidence_invariance, input_dependency_score, composite_leakage_score
from src.display_utils import show_llm_response, show_comparison, show_timeliness_leakage
import json
import re
import numpy as np
import pandas as pd

set_seed()
(PROJECT_ROOT / "data" / "results").mkdir(parents=True, exist_ok=True)

## 1. 加载数据 + 解析工具函数

In [ ]:
test_cases = load_test_cases()
variants = load_counterfactual_variants()

variant_map = {}
for v in variants:
    variant_map.setdefault(v.original_case_id, {})[v.variant_type] = v

client = LLMClient()
print(f"Test cases: {len(test_cases)}, Variants: {len(variants)}")

In [ ]:
def parse_binary(raw: str) -> dict:
    data = extract_json_robust(raw)
    if data and "direction" in data:
        d = data["direction"].lower()
        direction = d if d in ("up", "down") else "neutral"
    else:
        direction = "neutral"
    return {"direction": direction, "confidence": 1.0 if direction != "neutral" else 0.5, "distribution": None}


def parse_scalar(raw: str) -> dict:
    data = extract_json_robust(raw)
    if data and "score" in data:
        score = float(data["score"])
    else:
        score = 0.0
    direction = "up" if score > 0.1 else ("down" if score < -0.1 else "neutral")
    return {"direction": direction, "confidence": abs(score), "distribution": None}


def parse_5bin(raw: str) -> dict:
    data = extract_json_robust(raw)
    keys = ["strong_bear", "weak_bear", "neutral", "weak_bull", "strong_bull"]
    if not data or not all(k in data for k in keys):
        return {"direction": "neutral", "confidence": 0.5, "distribution": [0.2]*5}
    dist = [float(data[k]) for k in keys]
    total = sum(dist)
    dist = [d / total for d in dist] if total > 0 else [0.2]*5
    bull, bear = dist[3] + dist[4], dist[0] + dist[1]
    direction = "up" if bull > bear + 0.1 else ("down" if bear > bull + 0.1 else "neutral")
    return {"direction": direction, "confidence": max(bull, bear, dist[2]), "distribution": dist}


def _parse_judge(raw: str) -> dict:
    """解析 LLM-as-judge 的方向判断响应。"""
    data = extract_json_robust(raw)
    if data and "direction" in data:
        return {"direction": data["direction"], "confidence": float(data.get("confidence", 0.5)), "distribution": None}
    return {"direction": "neutral", "confidence": 0.5, "distribution": None}


PARSERS = {
    "binary": parse_binary,
    "scalar": parse_scalar,
    "5-bin": parse_5bin,
}

print("解析函数就绪")

## 2. 四种格式 × 反事实攻击

使用 baseline（无掩码）策略，对比四种输出格式下的 PC/CI/IDS。

In [ ]:
config = MaskingConfig()  # baseline, no masking
formats = ["binary", "scalar", "5-bin", "free-text"]

all_results = []

for fmt in formats:
    print(f"\n{'='*50}")
    print(f"Output format: {fmt}")
    print(f"{'='*50}")

    # 收集 orig + cf 文本对
    tasks = []  # (tc, vt_name)
    orig_texts = []
    cf_texts = []
    for tc in test_cases:
        for vt_name, variant in variant_map.get(tc.id, {}).items():
            tasks.append((tc, vt_name))
            orig_texts.append(tc.news.content)
            cf_texts.append(variant.modified_content)

    # 并发 scoring
    all_texts = orig_texts + cf_texts
    all_responses = run_scoring_batch(client, config, all_texts, fmt)
    orig_responses = all_responses[:len(tasks)]
    cf_responses = all_responses[len(tasks):]

    # free-text 需要额外 judge 调用
    if fmt == "free-text":
        judge_sys = "从以下分析文本中提取市场方向判断。输出JSON：{\"direction\": \"up\"/\"down\"/\"neutral\", \"confidence\": 0-1}"
        judge_prompts = [(judge_sys, r.raw_response) for r in orig_responses] + \
                        [(judge_sys, r.raw_response) for r in cf_responses]
        print(f"  free-text judge 并发调用 {len(judge_prompts)} 个请求 ...")
        judge_responses = client.batch_chat_concurrent(judge_prompts)
        orig_judge = judge_responses[:len(tasks)]
        cf_judge = judge_responses[len(tasks):]

        for (tc, vt_name), oj_resp, cj_resp in zip(tasks, orig_judge, cf_judge):
            orig_parsed = _parse_judge(oj_resp.raw_response)
            cf_parsed = _parse_judge(cj_resp.raw_response)
            all_results.append({
                "format": fmt, "case_id": tc.id, "variant_type": vt_name,
                "orig_direction": orig_parsed["direction"], "cf_direction": cf_parsed["direction"],
                "orig_confidence": orig_parsed["confidence"], "cf_confidence": cf_parsed["confidence"],
                "orig_distribution": None, "cf_distribution": None,
            })
    else:
        parser = PARSERS[fmt]
        for (tc, vt_name), orig_resp, cf_resp in zip(tasks, orig_responses, cf_responses):
            orig_parsed = parser(orig_resp.raw_response)
            cf_parsed = parser(cf_resp.raw_response)
            all_results.append({
                "format": fmt, "case_id": tc.id, "variant_type": vt_name,
                "orig_direction": orig_parsed["direction"], "cf_direction": cf_parsed["direction"],
                "orig_confidence": orig_parsed["confidence"], "cf_confidence": cf_parsed["confidence"],
                "orig_distribution": orig_parsed["distribution"], "cf_distribution": cf_parsed["distribution"],
            })

print(f"\nTotal results: {len(all_results)}")

## 3. 计算各格式的 PC / CI / IDS

注意：IDS 仅在 5-bin 格式下有自然的概率分布，其他格式用方向+置信度近似。

In [ ]:
df = pd.DataFrame(all_results)

metrics_rows = []
for (fmt, vtype), group in df.groupby(["format", "variant_type"]):
    orig_dirs = group["orig_direction"].tolist()
    cf_dirs = group["cf_direction"].tolist()
    orig_confs = group["orig_confidence"].tolist()
    cf_confs = group["cf_confidence"].tolist()

    pc = prediction_consistency(orig_dirs, cf_dirs)
    consistent = [o == c for o, c in zip(orig_dirs, cf_dirs)]
    ci = confidence_invariance(orig_confs, cf_confs, consistent)

    # IDS: 仅 5-bin 有真实分布，其他格式用置信度构造伪分布
    orig_dists = group["orig_distribution"].tolist()
    cf_dists = group["cf_distribution"].tolist()
    if fmt == "5-bin" and orig_dists[0] is not None:
        ids = input_dependency_score(orig_dists, cf_dists)
    else:
        # 用 [1-conf, conf] 作为二元伪分布
        pseudo_orig = [[1 - c, c] for c in orig_confs]
        pseudo_cf = [[1 - c, c] for c in cf_confs]
        ids = input_dependency_score(pseudo_orig, pseudo_cf)

    L = composite_leakage_score(pc, ci, ids)
    metrics_rows.append({
        "format": fmt, "variant_type": vtype,
        "PC": pc, "CI": ci, "IDS": ids, "L": L, "n": len(group),
    })

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string(index=False, float_format="%.3f"))

### 各格式指标解读

**关键发现：IDS 差异悬殊。**

- **5-bin 格式** IDS ≈ 1.0：5 桶概率分布为 KL 散度提供了丰富的信号。即使两次预测方向相同（如都是"看涨"），只要概率分布形状不同（如 [5,10,15,40,30] vs [5,10,20,35,30]），IDS 也能捕捉到差异。
- **binary 格式** IDS = 0：只有涨/跌两个值，无法计算 KL 散度。PC 也因随机一致率高（50%）而灵敏度最低。相当于用抛硬币来检测泄露。
- **scalar 格式** IDS ≈ 0.15：一维连续值比二值好，但信息量远不如 5 维分布。
- **free-text 格式** IDS ≈ 0.03：方向由 judge 提取后退化为类似 binary 的信号，但 free-text 的独特价值在于可检测质性泄露（见下文）。

**结论：** 如果目标是量化泄露程度，5-bin 格式是最优选择。如果目标是发现具体的泄露证据（如模型引用了新闻发布后的事件），free-text 格式不可替代。

## 4. 可视化：格式 × 指标对比

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 按格式汇总（跨 variant_type 取均值）
fmt_summary = metrics_df.groupby("format")[["PC", "CI", "IDS", "L"]].mean()
fmt_summary = fmt_summary.loc[["binary", "scalar", "5-bin", "free-text"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：PC / CI / IDS 分组柱状图
fmt_summary[["PC", "CI", "IDS"]].plot(kind="bar", ax=axes[0],
    color=["#F44336", "#FF9800", "#4CAF50"])
axes[0].set_title("各输出格式的泄露指标均值")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc="upper right")

# 右图：综合泄露分数 L
fmt_summary["L"].plot(kind="bar", ax=axes[1], color="#9C27B0")
axes[1].set_title("综合泄露分数 L (越低越好)")
axes[1].set_ylabel("L = 0.4·PC + 0.3·CI - 0.3·IDS")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'output_format_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 热力图：格式 × 变体类型 的综合泄露分数
pivot_L = metrics_df.pivot(index="format", columns="variant_type", values="L")
pivot_L = pivot_L.loc[["binary", "scalar", "5-bin", "free-text"]]

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_L, annot=True, fmt=".3f", cmap="RdYlGn_r", center=0.3)
plt.title("综合泄露分数 L：输出格式 × 扰动类型")
plt.ylabel("输出格式")
plt.xlabel("扰动类型")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'output_format_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Free-text 格式：未来信息泄露检测

free-text 格式可能暴露最多质性泄露信号（如直接提及后续事件）。用 timeliness judge 检测。

In [ ]:
# Cell 5a: 计算 — Phase 1/2/3 (API 调用 + 解析)
tc_map = {tc.id: tc for tc in test_cases}

# Phase 1: 并发获取所有 free-text 分析
freetext_config = MaskingConfig()
ft_prompts = [scoring_prompt(tc.news.content, freetext_config, "free-text") for tc in test_cases]
print(f"并发调用 {len(ft_prompts)} 个 free-text 请求 ...")
ft_responses = client.batch_chat_concurrent(ft_prompts)

# Phase 2: 并发调用 timeliness judge（传入原始新闻全文以区分"复述输入"和"注入未来知识"）
judge_prompts = []
for tc, resp in zip(test_cases, ft_responses):
    news_date = tc.news.publish_time.strftime("%Y-%m-%d")
    judge_sys, judge_usr = timeliness_judge_prompt(news_date, tc.news.content, resp.raw_response)
    judge_prompts.append((judge_sys, judge_usr))

print(f"并发调用 {len(judge_prompts)} 个 timeliness judge 请求 ...")
judge_responses = client.batch_chat_concurrent(judge_prompts)

# Phase 3: 解析结果，构建展示数据
timeliness_results = []
timeliness_display_data = []
for tc, resp, judge_resp in zip(test_cases, ft_responses, judge_responses):
    news_date = tc.news.publish_time.strftime("%Y-%m-%d")
    judge_data = extract_json_robust(judge_resp.raw_response)
    has_future = judge_data.get("has_future_info", False) if judge_data else False
    severity = judge_data.get("severity", "none") if judge_data else "none"
    evidence = judge_data.get("evidence", []) if judge_data else []

    row = {
        "case_id": tc.id,
        "news_date": news_date,
        "has_future_info": has_future,
        "severity": severity,
        "evidence": evidence,
        "response_text": resp.raw_response,
        "judge_text": judge_resp.raw_response,
    }
    timeliness_results.append(row)

    timeliness_display_data.append({
        **row,
        "news_title": tc.news.title,
        "news_content": tc.news.content,
        "known_outcome": tc.known_outcome,
        "outcome_date": tc.outcome_date,
    })

future_leak_rate = sum(1 for r in timeliness_results if r["has_future_info"]) / len(timeliness_results)
print(f"\nFree-text 未来信息泄露率: {future_leak_rate:.1%}")
print(f"  major: {sum(1 for r in timeliness_results if r['severity'] == 'major')}")
print(f"  minor: {sum(1 for r in timeliness_results if r['severity'] == 'minor')}")
print(f"  none:  {sum(1 for r in timeliness_results if r['severity'] == 'none')}")

In [ ]:
import random

# Cell 5b: 随机抽取展示泄露案例（重新运行刷新样本）
leaked = [d for d in timeliness_display_data if d["has_future_info"]]
non_leaked = [d for d in timeliness_display_data if not d["has_future_info"]]

# 展示全部泄露案例 + 随机抽取部分无泄露案例，合计最多 10 条
n_non_leaked = max(0, 10 - len(leaked))
display_items = leaked + random.sample(non_leaked, min(n_non_leaked, len(non_leaked)))

print(f"展示 {len(leaked)} 条泄露 + {len(display_items) - len(leaked)} 条无泄露（共 {len(display_items)} 条）")
for d in display_items:
    show_timeliness_leakage(
        case_id=d["case_id"],
        news_title=d["news_title"],
        news_date=d["news_date"],
        news_content=d["news_content"],
        response_text=d["response_text"],
        evidence=d["evidence"],
        known_outcome=d["known_outcome"],
        outcome_date=d["outcome_date"],
        severity=d["severity"],
    )

In [ ]:
# Cell 5c: 汇总统计（不再全量展示 42 行表格）
summary_rows = [
    {
        "case_id": d["case_id"],
        "news_date": d["news_date"],
        "severity": d["severity"],
        "has_future_info": d["has_future_info"],
    }
    for d in timeliness_display_data
]
summary_df = pd.DataFrame(summary_rows)
print(f"Timeliness 汇总 ({len(summary_df)} 条)：")
print(summary_df["severity"].value_counts().to_string())

### 未来信息泄露检测解读

**Free-text 格式独特价值：** 在结构化格式（binary/scalar/5-bin）中，泄露表现为"数字不该那么自信"或"方向不该一致"——这些是统计信号，无法具体指出泄露了什么。Free-text 格式则可以直接在文本中发现"这条分析引用了新闻发布后才发生的事件"。

**Timeliness Judge 的工作方式：** 给定（新闻日期、新闻原文、模型分析），检查分析中是否包含新闻发布后才可知的信息。关键改进：传入了原始新闻全文，使 judge 能区分"模型复述了新闻中已有的信息"（不算泄露）和"模型引用了新闻未提及的后续事件"（真泄露）。

上方卡片中：
- **红色/橙色边框（minor/major）** 的案例：judge 认为模型分析中包含了未来信息。
- **绿色边框（none）** 的案例：模型分析未引用未来信息。
- "泄露证据"栏列出了 judge 识别的可疑语句，黄色高亮显示在模型分析文本中。

## 6. 结论与建议

In [ ]:
# 灵敏度排名：哪种格式最容易检出泄露？
print("各格式泄露检测灵敏度分析：")
print("="*50)
for fmt in ["binary", "scalar", "5-bin", "free-text"]:
    row = fmt_summary.loc[fmt]
    print(f"\n{fmt}:")
    print(f"  PC={row['PC']:.3f}  CI={row['CI']:.3f}  IDS={row['IDS']:.3f}  L={row['L']:.3f}")

best_fmt = fmt_summary["L"].idxmin()
worst_fmt = fmt_summary["L"].idxmax()
print(f"\n{'='*50}")
print(f"泄露最低（最佳消除）: {best_fmt} (L={fmt_summary.loc[best_fmt, 'L']:.3f})")
print(f"泄露最高（最易暴露）: {worst_fmt} (L={fmt_summary.loc[worst_fmt, 'L']:.3f})")
print(f"Free-text 未来信息泄露率: {future_leak_rate:.1%}")

print(f"\n给 Thales 的建议：")
print(f"  - 5-bin 格式在 IDS 维度提供最丰富的信号（KL 散度捕捉分布形状变化）")
print(f"  - binary 格式随机一致率 50%，PC 灵敏度最低")
print(f"  - free-text 格式可暴露质性泄露（直接提及后续事件），适合深度审计")

### 格式选择解读

- **5-bin 格式 L 最低**：因为 5 桶概率分布提供了最丰富的 IDS 信号——KL 散度能捕捉概率分布的细微形状变化，即使方向未变，分布也会发生位移。这使得 IDS 项能有效抵消 PC 和 CI 的贡献。
- **binary 格式 L 最高**：二分类只有涨/跌两个值，随机一致率已达 50%，且无概率分布可供 IDS 计算，泄露信号被格式噪声淹没。
- **free-text 格式的独特价值**：虽然 L 不是最低，但它能暴露质性泄露（直接提及后续事件），这是其他格式无法检测的。timeliness judge 现已传入原始新闻全文，能区分"复述输入内容"和"注入未来知识"，减少误判。
- **后续 notebook 统一使用 5-bin 格式**作为主要评估格式。

In [ ]:
# 保存所有结果
output = {
    "format_metrics": metrics_rows,
    "format_summary": fmt_summary.to_dict(),
    "timeliness": timeliness_results,
    "future_leak_rate": future_leak_rate,
}
output_path = PROJECT_ROOT / 'data' / 'results' / 'output_format_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"Results saved to {output_path}")